In [5]:
import pandas as pd
import numpy as np
from faker import Faker
from pathlib import Path
import random
from datetime import datetime, timedelta

fake = Faker()
np.random.seed(42)
random.seed(42)

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DATA_DIR = BASE_DIR / "data" / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR)

print("Raw data directory:", RAW_DATA_DIR)

print("Processed data directory:", PROCESSED_DATA_DIR)











Base directory: c:\Users\samis\population-health-risk-scoring-anaytics-engineering
Raw data directory: c:\Users\samis\population-health-risk-scoring-anaytics-engineering\data\raw
Processed data directory: c:\Users\samis\population-health-risk-scoring-anaytics-engineering\data\processed


In [6]:
num_members = 5000

members = pd.DataFrame({
    "member_id": range(100000, 100000 + num_members),

    "first_name": [fake.first_name() for _ in range(num_members)],
    "last_name": [fake.last_name() for _ in range(num_members)],

    "gender": np.random.choice(
        ["Male", "Female"],
        size=num_members,
        p=[0.48, 0.52]
    ),

    "date_of_birth": [
        fake.date_between(start_date='-90y', end_date='-18y')
        for _ in range(num_members)
    ],

    "state": np.random.choice(
        ["FL", "TX", "CA", "NY", "GA"],
        size=num_members
    ),

    "plan_type": np.random.choice(
        ["Commercial", "Medicare Advantage", "Medicaid"],
        size=num_members,
        p=[0.55, 0.25, 0.20]
    ),

    "enrollment_date": [
        fake.date_between(start_date='-5y', end_date='today')
        for _ in range(num_members)
    ]
})

members.head()

,member_id,first_name,last_name,gender,date_of_birth,state,plan_type,enrollment_date
0,100000,Margaret,Perez,Male,1956-01-14,CA,Medicare Advantage,2026-02-20
1,100001,Leslie,Lawson,Female,1979-12-19,CA,Commercial,2022-02-01
2,100002,Katherine,Smith,Female,1976-08-18,NY,Commercial,2024-12-02
3,100003,Donna,Johnson,Female,2007-01-08,CA,Medicare Advantage,2022-09-01
4,100004,Melissa,Leblanc,Male,1976-11-15,GA,Commercial,2022-06-16


In [7]:
chronic_conditions = [
    "Diabetes",
    "CHF",
    "COPD",
    "Hypertension",
    "CKD",
    "Asthma",
    "Depression",
    "Obesity"
]

member_conditions = []

for member_id in members["member_id"]:

    num_conditions = np.random.choice(
        [0, 1, 2, 3],
        p=[0.35, 0.40, 0.20, 0.05]
    )

    assigned_conditions = random.sample(
        chronic_conditions,
        k=num_conditions
    )

    for condition in assigned_conditions:

        member_conditions.append({
            "member_id": member_id,
            "condition_name": condition,

            "risk_level": np.random.choice(
                ["Low", "Medium", "High"],
                p=[0.50, 0.35, 0.15]
            ),

            "diagnosis_date": fake.date_between(
                start_date='-5y',
                end_date='today'
            )
        })

conditions_df = pd.DataFrame(member_conditions)

conditions_df.head()

,member_id,condition_name,risk_level,diagnosis_date
0,100000,CHF,Low,2023-06-29
1,100001,Diabetes,Medium,2025-02-18
2,100002,CKD,Medium,2024-03-07
3,100003,Hypertension,Medium,2024-10-28
4,100003,CHF,Low,2023-05-06


In [8]:
claim_types = [
    "Inpatient",
    "Outpatient",
    "Emergency",
    "Primary Care",
    "Specialist",
    "Pharmacy"
]

claims = []

claim_id = 500000

for member_id in members["member_id"]:

    num_claims = np.random.randint(1, 15)

    for _ in range(num_claims):

        service_date = fake.date_between(
            start_date='-2y',
            end_date='today'
        )

        claim_type = np.random.choice(
            claim_types,
            p=[0.10, 0.25, 0.15, 0.25, 0.15, 0.10]
        )

        allowed_amount = round(
            np.random.gamma(2, 500),
            2
        )

        paid_amount = round(
            allowed_amount * np.random.uniform(0.70, 0.98),
            2
        )

        claims.append({
            "claim_id": claim_id,
            "member_id": member_id,
            "claim_type": claim_type,
            "service_date": service_date,
            "allowed_amount": allowed_amount,
            "paid_amount": paid_amount,
            "admit_flag": 1 if claim_type == "Inpatient" else 0
        })

        claim_id += 1

claims_df = pd.DataFrame(claims)

claims_df.head()

,claim_id,member_id,claim_type,service_date,allowed_amount,paid_amount,admit_flag
0,500000,100000,Primary Care,2025-07-04,745.58,635.12,0
1,500001,100000,Outpatient,2024-10-13,1180.12,872.90,0
2,500002,100000,Emergency,2025-09-27,804.82,714.38,0
3,500003,100001,Pharmacy,2024-07-25,606.74,551.12,0
4,500004,100001,Outpatient,2025-06-23,1258.62,1131.93,0


In [9]:
interaction_types = [
    "Phone Call",
    "Care Management",
    "Email Outreach",
    "SMS Outreach"
]

interaction_statuses = [
    "Successful",
    "Unable to Reach",
    "Declined",
    "Left Voicemail"
]

interactions = []

interaction_id = 900000

engaged_members = np.random.choice(
    members["member_id"],
    size=int(num_members * 0.55),
    replace=False
)

for member_id in members["member_id"]:

    num_interactions = np.random.randint(0, 6)

    for _ in range(num_interactions):

        if member_id in engaged_members:
            status = np.random.choice(
                interaction_statuses,
                p=[0.65, 0.15, 0.10, 0.10]
            )
        else:
            status = np.random.choice(
                interaction_statuses,
                p=[0.10, 0.45, 0.15, 0.30]
            )

        interactions.append({
            "interaction_id": interaction_id,
            "member_id": member_id,

            "interaction_type": np.random.choice(
                interaction_types
            ),

            "interaction_status": status,

            "interaction_date": fake.date_between(
                start_date='-1y',
                end_date='today'
            )
        })

        interaction_id += 1

interactions_df = pd.DataFrame(interactions)

interactions_df.head()

,interaction_id,member_id,interaction_type,interaction_status,interaction_date
0,900000,100000,Email Outreach,Unable to Reach,2026-04-05
1,900001,100000,Care Management,Successful,2025-11-24
2,900002,100000,Care Management,Unable to Reach,2025-07-01
3,900003,100000,SMS Outreach,Unable to Reach,2025-08-05
4,900004,100001,SMS Outreach,Unable to Reach,2026-02-04


In [10]:
members.to_csv(RAW_DATA_DIR / "members.csv", index=False)

conditions_df.to_csv(RAW_DATA_DIR / "conditions.csv", index=False)

claims_df.to_csv(RAW_DATA_DIR / "claims.csv", index=False)

interactions_df.to_csv(RAW_DATA_DIR / "interactions.csv", index=False)


print("Files Saved")
print(RAW_DATA_DIR / "members.csv")
print(RAW_DATA_DIR / "conditions.csv")
print(RAW_DATA_DIR / "claims.csv")
print(RAW_DATA_DIR / "interactions.csv")



Files Saved
c:\Users\samis\population-health-risk-scoring-anaytics-engineering\data\raw\members.csv
c:\Users\samis\population-health-risk-scoring-anaytics-engineering\data\raw\conditions.csv
c:\Users\samis\population-health-risk-scoring-anaytics-engineering\data\raw\claims.csv
c:\Users\samis\population-health-risk-scoring-anaytics-engineering\data\raw\interactions.csv
